# Telecom Customer Churn Prediction
**Random Forest vs. Artificial Neural Network (scikit-learn + TensorFlow/Keras)**

Pipeline: data loading -> EDA -> cleaning -> feature engineering -> preprocessing -> model training & tuning ->
model comparison -> artifact export (consumed by the Flask app in `app.py`).

**Dataset:** IBM *Telco Customer Churn* (7,043 customers, 21 columns). Download `WA_Fn-UseC_-Telco-Customer-Churn.csv`
from Kaggle (search "Telco Customer Churn") and place it in `data/`.

> All cleaning / feature engineering lives in `src/features.py` and is re-used by the Flask app, so training and serving
> apply identical transformations.

In [ ]:
import json
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import features as ft  # shared with the Flask app

SEED = 42
np.random.seed(SEED)
DATA_PATH = ROOT / "data" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)

## 1. Load data

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Place the Telco churn CSV at {DATA_PATH}")
raw = pd.read_csv(DATA_PATH)
print("Shape:", raw.shape)
raw.head()

## 2. Exploratory data analysis

In [ ]:
blank_total = (raw["TotalCharges"].astype(str).str.strip() == "").sum()
print(f"Blank TotalCharges (new customers, tenure=0): {blank_total}")
print("Missing values per column:", int(raw.isna().sum().sum()))
print("\nChurn rate:", round((raw["Churn"] == "Yes").mean() * 100, 1), "%  <- imbalanced target")

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
raw["Churn"].value_counts().plot.bar(ax=ax[0], color=["#4c78a8", "#e45756"], rot=0, title="Churn distribution")
raw.groupby("Contract")["Churn"].apply(lambda s: (s == "Yes").mean() * 100).plot.bar(
    ax=ax[1], rot=0, color="#e45756", title="Churn rate (%) by contract")
sns.histplot(data=raw, x="tenure", hue="Churn", multiple="stack", bins=30, ax=ax[2])
ax[2].set_title("Tenure by churn")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
raw.groupby("InternetService")["Churn"].apply(lambda s: (s == "Yes").mean() * 100).plot.bar(
    ax=ax[0], rot=0, color="#f58518", title="Churn rate (%) by internet service")
raw.groupby("PaymentMethod")["Churn"].apply(lambda s: (s == "Yes").mean() * 100).plot.bar(
    ax=ax[1], rot=20, color="#54a24b", title="Churn rate (%) by payment method")
sns.boxplot(data=raw, x="Churn", y="MonthlyCharges", ax=ax[2])
ax[2].set_title("Monthly charges by churn")
plt.tight_layout()
plt.show()

## 3. Cleaning & feature engineering
**Cleaning** (`ft.clean`): drop `customerID`, convert `TotalCharges` to numeric (blank -> `tenure x MonthlyCharges`),
collapse *"No internet service" / "No phone service"* into *"No"*.

**Engineered features** (`ft.engineer`):
| Feature | Meaning |
|---|---|
| `NumServices` | number of subscribed services |
| `NumProtectionServices` | security / backup / device protection / tech support count |
| `AvgMonthlySpend` | `TotalCharges / tenure` - historical average bill |
| `ChargeDelta` | current monthly charge minus historical average (price-hike signal) |
| `TenureGroup` | tenure bucket (0-12m, 13-24m, ...) |
| `AutoPay` | pays through automatic bank transfer / credit card |
| `IsMonthToMonth` | no long-term contract |

In [ ]:
y = ft.encode_target(raw[ft.TARGET])
df = ft.engineer(ft.clean(raw.drop(columns=[ft.TARGET])))
X = df[ft.MODEL_FEATURES]
print("Feature matrix:", X.shape, "| churn rate:", round(y.mean(), 3))
df[ft.ENGINEERED_NUMERIC + ["TenureGroup"]].head()

In [ ]:
corr = X[ft.NUMERIC_FEATURES].assign(Churn=y).corr()["Churn"].drop("Churn").sort_values()
corr.plot.barh(figsize=(7, 5), color=np.where(corr > 0, "#e45756", "#4c78a8"),
               title="Correlation of numeric features with churn")
plt.show()

## 4. Train/test split & preprocessing
Stratified 80/20 split. The preprocessor (StandardScaler for numerics + One-Hot for categoricals) is **fit on the
training set only** to avoid data leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

preprocessor = ft.build_preprocessor()
Xtr = preprocessor.fit_transform(X_train)
Xte = preprocessor.transform(X_test)
feature_names = [n.split("__", 1)[1] for n in preprocessor.get_feature_names_out()]
print(f"Train: {Xtr.shape} | Test: {Xte.shape} | features after encoding: {len(feature_names)}")

### Evaluation helper & baseline
Because ~73% of customers do **not** churn, a model that always predicts "stay" is already ~73% accurate.
Accuracy alone is therefore misleading - we also track precision, recall, F1 and ROC-AUC.

In [ ]:
results, probas = {}, {}

def evaluate(name, proba, threshold=0.5, verbose=True):
    pred = (proba >= threshold).astype(int)
    results[name] = {
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
    }
    probas[name] = proba
    if verbose:
        print(f"=== {name} (threshold={threshold}) ===")
        print(classification_report(y_test, pred, target_names=["Stay", "Churn"], digits=3))
    return pred

baseline = DummyClassifier(strategy="most_frequent").fit(Xtr, y_train)
print("Majority-class baseline accuracy:", round(accuracy_score(y_test, baseline.predict(Xte)), 4))

## 5. Random Forest

In [ ]:
rf_default = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1).fit(Xtr, y_train)
_ = evaluate("RF (default)", rf_default.predict_proba(Xte)[:, 1], verbose=False)
results["RF (default)"]

In [ ]:
param_dist = {
    "n_estimators": [200, 300, 500],
    "max_depth": [6, 8, 10, 12, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2", 0.3],
}
search = RandomizedSearchCV(
    RandomForestClassifier(random_state=SEED, n_jobs=1),
    param_distributions=param_dist, n_iter=20, scoring="roc_auc",
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    random_state=SEED, n_jobs=-1, verbose=1,
)
search.fit(Xtr, y_train)
rf = search.best_estimator_
print("Best CV ROC-AUC:", round(search.best_score_, 4))
print("Best params:", search.best_params_)

In [ ]:
_ = evaluate("Random Forest", rf.predict_proba(Xte)[:, 1])

### Decision-threshold analysis
The default 0.5 cut-off maximises accuracy, but a retention team usually cares more about **catching churners** (recall).
Lowering the threshold trades accuracy/precision for recall - this is a business decision, not a modelling one.

In [ ]:
rows = []
for t in np.arange(0.20, 0.71, 0.05):
    p = (probas["Random Forest"] >= t).astype(int)
    rows.append({"threshold": round(t, 2), "accuracy": accuracy_score(y_test, p),
                 "precision": precision_score(y_test, p, zero_division=0),
                 "recall": recall_score(y_test, p), "f1": f1_score(y_test, p)})
pd.DataFrame(rows).round(3).set_index("threshold")

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values().tail(15)
importances.plot.barh(figsize=(8, 6), color="#4c78a8", title="Random Forest - top 15 feature importances")
plt.show()

## 6. Artificial Neural Network (TensorFlow / Keras)
Architecture: `Dense(64, relu) -> Dropout(0.3) -> Dense(32, relu) -> Dropout(0.2) -> Dense(1, sigmoid)`,
Adam optimiser, binary cross-entropy, early stopping on validation loss, LR reduction on plateau.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(SEED)

ann = keras.Sequential([
    layers.Input(shape=(Xtr.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
])
ann.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
            metrics=["accuracy", keras.metrics.AUC(name="auc")])
ann.summary()

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5),
]
history = ann.fit(Xtr.astype("float32"), y_train.values, validation_split=0.2,
                  epochs=150, batch_size=32, callbacks=callbacks, verbose=2)
ann.save(MODEL_DIR / "ann_model.keras")

In [ ]:
hist = pd.DataFrame(history.history)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
hist[["loss", "val_loss"]].plot(ax=ax[0], title="Loss")
hist[["accuracy", "val_accuracy"]].plot(ax=ax[1], title="Accuracy")
plt.tight_layout()
plt.show()

In [ ]:
ann_proba = ann.predict(Xte.astype("float32"), verbose=0).ravel()
_ = evaluate("ANN", ann_proba)

## 7. Model comparison

In [ ]:
comparison = pd.DataFrame(results).T.round(4).sort_values("ROC-AUC", ascending=False)
display(comparison)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
for name, p in probas.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    ax[0].plot(fpr, tpr, label=f"{name} (AUC={results[name]['ROC-AUC']:.3f})")
ax[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
ax[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC curves")
ax[0].legend()
comparison[["Accuracy", "Precision", "Recall", "F1"]].plot.bar(ax=ax[1], rot=15, title="Metrics at threshold 0.5")
plt.tight_layout()
plt.show()

In [ ]:
final = {k: v for k, v in probas.items() if k in ("Random Forest", "ANN")}
fig, axes = plt.subplots(1, len(final), figsize=(5.5 * len(final), 4.5))
for ax, (name, p) in zip(np.atleast_1d(axes), final.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, (p >= 0.5).astype(int), display_labels=["Stay", "Churn"],
                                            ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 8. Export artifacts for the Flask app
`preprocessor.joblib`, `rf_model.joblib`, `ann_model.keras` (already saved above) and `metrics.json` go to `models/`.

In [ ]:
joblib.dump(preprocessor, MODEL_DIR / "preprocessor.joblib")
joblib.dump(rf, MODEL_DIR / "rf_model.joblib", compress=3)

metrics = {
    "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "n_train": int(len(X_train)), "n_test": int(len(X_test)),
    "majority_baseline_accuracy": round(float(accuracy_score(y_test, baseline.predict(Xte))), 4),
    "rf_best_params": search.best_params_,
    "test_metrics": {k: {m: round(float(v), 4) for m, v in r.items()}
                     for k, r in results.items() if k in ("Random Forest", "ANN")},
}
(MODEL_DIR / "metrics.json").write_text(
    json.dumps(metrics, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
print(json.dumps(metrics["test_metrics"], indent=2))
print("Saved:", sorted(p.name for p in MODEL_DIR.iterdir() if p.name != ".gitkeep"))

### Sanity check - run the exact code path the Flask app uses

In [ ]:
from predictor import ChurnPredictor, validate_customer

predictor = ChurnPredictor(MODEL_DIR)
print("Loaded models:", predictor.available_models)

high_risk = {"gender": "Female", "SeniorCitizen": 1, "Partner": "No", "Dependents": "No", "tenure": 3,
             "PhoneService": "Yes", "MultipleLines": "No", "InternetService": "Fiber optic",
             "OnlineSecurity": "No", "OnlineBackup": "No", "DeviceProtection": "No", "TechSupport": "No",
             "StreamingTV": "Yes", "StreamingMovies": "Yes", "Contract": "Month-to-month",
             "PaperlessBilling": "Yes", "PaymentMethod": "Electronic check", "MonthlyCharges": 89.9}
_, record = validate_customer(high_risk)
print(json.dumps(predictor.predict([record])[0], indent=2))

## 9. Summary
* Two models were trained on the same preprocessed features and compared on a held-out, stratified test set.
* Read the comparison table above for the final accuracy / precision / recall / F1 / ROC-AUC of each model.
* Start the API with `python app.py` (see `README.md`) - predictions and customer profiles are stored in MongoDB.

**Possible next steps:** SMOTE / class weights for higher recall, gradient boosting (XGBoost/LightGBM) as a stronger
baseline, SHAP explanations, probability calibration, scheduled retraining.